# 트리에서의 동적 계획법 (Dynamic Programming On Trees)

## 트리와 쿼리

- 문제 출처: [백준 15681번](https://www.acmicpc.net/problem/15681)

`-` 부모를 루트로 하는 서브 트리에 속한 정점의 수는 자식들을 루트로 하는 서브 트리에 속한 정점의 수들의 합에 $1$을 더한 것과 같다

`-` 부모 정점 $P$와 자식 정점 $X, Y$가 있다고 해보자

`-` 임의의 노드 $u$를 루트로 하는 서브 트리에 속한 정점의 수를 $f(u)$라고 하자

`-` 그럼 $f(P) = f(X) + f(Y) + 1$이다 ($1$을 더하는 건 부모 자기 자신이다)

`-` 트리 구조를 생각하면 위의 식은 자연스럽게 성립하니 이를 바탕으로 문제를 해결하자

`-` 리트 노드라면 함숫값은 $1$이다 (리트 노드를 루트로 하는 서브 트리에 속한 정점은 자기 자신밖에 없다)

In [10]:
import sys
from collections import defaultdict

sys.setrecursionlimit(10**5)


class Node:
    def __init__(self, key):
        self.key = key
        self.children = []

    def add_child(self, child_node):
        self.children.append(child_node)


def make_tree(root_node, graph):
    root_node = Node(root_node)
    stack = [root_node]
    visited = {root_node.key: True}
    while stack:
        node = stack.pop()
        for neighbor in graph[node.key]:
            if neighbor not in visited:
                neighbor = Node(neighbor)
                node.add_child(neighbor)
                visited[neighbor.key] = True
                stack.append(neighbor)
    return root_node  # 루트 노드에 tree의 정보가 담겨있다


def count_nodes_in_subtree(root_node):
    if root_node.key in DP:
        return DP[root_node.key]
    num_nodes = 1  # 자기 자신
    for child in root_node.children:
        num_nodes += count_nodes_in_subtree(child)
    DP[root_node.key] = num_nodes
    return num_nodes


def solution():
    global DP
    N, R, Q = map(int, input().split())
    graph = defaultdict(list)
    DP = {}
    for _ in range(N - 1):
        U, V = map(int, input().split())
        graph[U].append(V)
        graph[V].append(U)
    root_node = make_tree(R, graph)
    count_nodes_in_subtree(root_node)
    for _ in range(Q):
        U = int(input())
        print(DP[U])


solution()

# input
# 9 5 3
# 1 3
# 4 3
# 5 4
# 5 6
# 6 7
# 2 3
# 9 6
# 6 8
# 5
# 4
# 8

 9 5 3
 1 3
 4 3
 5 4
 5 6
 6 7
 2 3
 9 6
 6 8
 5


9


 4


4


 8


1


- 힌트에 설명된 코드 (챗지피티가 작성함)

`-` 불필요한 클래스 객체 필요 없이 한 번에 탐색으로 모든 쿼리에 대답할 수 있는 코드

`-` 깔끔하게 구현되었다 (한 번의 탐색으로 모든 쿼리에 대답할 수 있다)

`-` 양방향 그래프 구조에서 루트에서 내려가는 단방향 트리처럼 탐색을 어떻게 하나 싶었는데 자식이 부모와 다른 경우만 탐색하도록 구현했다

`-` 그래서 부모로 거슬러 올라가지 않고 자식만 방문할 수 있다

```python
import sys
from collections import defaultdict

sys.setrecursionlimit(10**5)


def dfs(node, parent):
    subtree_size[node] = 1  # 자기 자신 포함
    for child in tree[node]:
        if child == parent:
            continue
        subtree_size[node] += dfs(child, node)
    return subtree_size[node]


def solution():
    global tree, subtree_size
    N, R, Q = map(int, sys.stdin.readline().split())
    tree = defaultdict(list)
    subtree_size = {}
    for _ in range(N - 1):
        U, V = map(int, sys.stdin.readline().split())
        tree[U].append(V)
        tree[V].append(U)
    dfs(R, -1)  # 루트 R에서 시작하는 DFS
    for _ in range(Q):
        U = int(sys.stdin.readline())
        print(subtree_size[U])


solution()
```

## 우수 마을

- 문제 출처: [백준 1949번](https://www.acmicpc.net/problem/1949)

`-` $p$를 루트로 하는 서브 트리에 있는 우수 마을의 총 주민수를 고려하자

`-` 이때 $p$가 우수 마을이거나 $p$의 자식이 우수 마을이다

`-` 2번 조건에 의해 둘 다 우수 마을일 수 없고 3번 조건에 의해 둘 다 우수 마을이 아닐 수는 없다

`-` 2번 조건과 3번 조건을 충족하기 위해 $p$가 우수 마을인지와 $p$의 부모가 우수 마을인지를 한 번에 고려할 것이다 

`-` `dp[p][1][0]`을 $p$의 부모는 우수 마을이 아니고 $p$가 우수 마을일 때 $p$를 루트로 하는 서브 트리에 있는 우수 마을의 총 주민수라고 하자

`-` 배열의 $2$번째 값은 $p$가 우수 마을인지 아닌지를 뜻하며 배열의 $3$번째 값은 $p$의 부모가 우수 마을인지 아닌지를 뜻한다 

`-` $p$의 자식들을 $c_1,c_2,\cdots, c_n$라고 하자

`-` 그럼 아래의 점화식이 성립한다

`-` $\operatorname{dp}[p][1][0] = \operatorname{village}[p] + \operatorname{dp}[c_1][0][1] + \operatorname{dp}[c_2][0][1] + \cdots + \operatorname{dp}[c_n][0][1]$이다

`-` $\operatorname{dp}[p][0][1] = \max(\{\operatorname{dp}[c_1][s_1][0] + \operatorname{dp}[c_2][s_2][0] + \cdots + \operatorname{dp}[c_n][s_n][0],\cdots\})$이다 (단, $s_i$는 $0$ 또는 $1$)

`-` $\operatorname{dp}[p][0][0] = \max(\{\operatorname{dp}[c_1][s_1][0] + \operatorname{dp}[c_2][s_2][0] + \cdots + \operatorname{dp}[c_n][s_n][0],\cdots\})$이다 (단, $s_i$중 적어도 하나는 $1$)

`-` 점화식 $2$번째에선 각각의 최댓값을 선택하면 된다

`-` 점화식 $3$번째에선 먼저 각각의 최댓값을 선택한다

`-` 만약 $s_i$가 모두 $0$이라면 $s_i$를 $1$로 바꿨을 때 손실이 가장 적은 것을 $1$로 바꾼다

`-` 위의 점화식을 기반으로 배열을 채워갈 때 불가능한 경우는 제외해야 한다

`-` 리프 노드 $l$에 대해 종료 조건을 정의하자

`-` $\operatorname{dp}[l][1][0] = \operatorname{village}[l]$

`-` $\operatorname{dp}[l][0][1] = 0$

`-` $\operatorname{dp}[l][0][0] = 0$, 불가능한 경우이지만 형식을 맞추기 위함 (어차피 최댓값이 될 수 없어서 괜찮다)

`-` `village[x]`는 마을 $x$의 주민 수이다

In [3]:
import sys
from collections import defaultdict

sys.setrecursionlimit(10**5)


def dfs(tree, node, parent):
    dp[node][1][0] += villages[node]
    excellent_at_least = False
    min_gap = INF
    for child in tree[node]:
        if child == parent:
            continue
        dp1, dp2, dp3 = dfs(tree, child, node)
        if dp1 >= dp3:
            dp_max = dp1
            excellent_at_least = True
        else:
            dp_max = dp3
            dp_min = dp1
            min_gap = min(min_gap, dp_max - dp_min)
        dp[node][1][0] += dp2
        dp[node][0][1] += dp_max
        dp[node][0][0] += dp_max
    if min_gap == INF:
        min_gap = 0
    if not excellent_at_least:
        dp[node][0][0] -= min_gap
    return dp[node][1][0], dp[node][0][1], dp[node][0][0]


def solution():
    global villages, INF, dp
    N = int(input())
    villages = [0] + list(map(int, input().split()))
    tree = defaultdict(list)
    INF = float("inf")
    for _ in range(N - 1):
        u, v = map(int, input().split())
        tree[u].append(v)
        tree[v].append(u)
    dp = [[[0 for _ in range(2)] for _ in range(2)] for _ in range(N + 1)]  # dp[x][1][0]은 루트가 x, x는 우수, x의 부모는 우수 아님
    ROOT = 1
    NONE = -1
    dp1, dp2, dp3 = dfs(tree, ROOT, NONE)
    answer = max(dp1, dp2, dp3)
    print(answer)


solution()

# input
# 7
# 1000 3000 4000 1000 2000 2000 7000
# 1 2
# 2 3
# 4 3
# 4 5
# 6 2
# 6 7

 7
 1000 3000 4000 1000 2000 2000 7000
 1 2
 2 3
 4 3
 4 5
 6 2
 6 7


14000


## 사회망 서비스(SNS)

- 문제 출처: [백준 2533번](https://www.acmicpc.net/problem/2533)

`-` 어떤 노드가 얼리 아답터인지 아닌지는 그 노드의 부모와 자식이 얼리 아답터인지 아닌지에 영향을 받는다

`-` 리프 노드는 자식을 가지지 않으므로 부모의 얼리 아답터 유무만 판단해도 된다

`-` 어떤 노드를 루트로 하는 서브 트리에 속한 최소 얼리 아답터의 수를 고려하자

`-` `dp[u][1][0]`를 노드 $u$가 얼리 아답터이고 $u$의 부모가 얼리 아답터가 아닐 때 $u$가 루트인 서브 트리에 속한 최소 얼리 아답터의 수라고 하자

`-` $u$가 얼리 아답터라면 부모와 자식은 얼리 아답터이든 아니든 상관 없다

`-` $u$가 얼리 아답터가 아니라면 부모와 자식은 얼리 아답터여야 한다

`-` 노드 $u$의 자식들을 $c_1, c_2, \cdots, c_u$라고 할 때 다음의 점화식이 성립한다

`-` $\operatorname{dp}[u][1][0] = \operatorname{dp}[u][1][1] = \min(\operatorname{dp}[c_1][0][1], \operatorname{dp}[c_1][1][1]) + \min(\operatorname{dp}[c_2][0][1], \operatorname{dp}[c_2][1][1]) + \cdots +  + \min(\operatorname{dp}[c_u][0][1], \operatorname{dp}[c_u][1][1])$

`-` $\operatorname{dp}[u][0][1] = \operatorname{dp}[c_1][1][0] +\operatorname{dp}[c_2][1][0]  + \cdots + \operatorname{dp}[c_u][1][0]$

`-` 또한, 리프 노드 $l$에 대해 $\operatorname{dp}[l][0][1] = 0,\; \operatorname{dp}[l][1][0] = \operatorname{dp}[l][1][1] = 1$이다

`-` 결과적으로 $\min(\operatorname{dp}[u][0][1], \operatorname{dp}[u][1][0], \operatorname{dp}[u][1][1])$이 정답이 된다

In [3]:
import sys
from collections import defaultdict

sys.setrecursionlimit(10**6)


def dfs(tree, node, parent):
    dp[node][0][1] = 0
    dp[node][1][0] = 1
    dp[node][1][1] = 1
    for child in tree[node]:
        if child == parent:
            continue
        dfs(tree, child, node)
        dp[node][0][1] += dp[child][1][0]
        dp[node][1][0] += min(dp[child][0][1], dp[child][1][1])
        dp[node][1][1] += min(dp[child][0][1], dp[child][1][1])


def solution():
    global dp
    N = int(input())
    tree = defaultdict(list)
    for _ in range(N - 1):
        u, v = map(int, input().split())
        tree[u].append(v)
        tree[v].append(u)
    dp = [[[0 for _ in range(2)] for _ in range(2)] for _ in range(N + 1)]
    ROOT_NODE = 1
    NONE = -1
    dfs(tree, ROOT_NODE, NONE)
    answer = min(dp[ROOT_NODE][0][1], dp[ROOT_NODE][1][0], dp[ROOT_NODE][1][1])
    print(answer)


solution()

# input
# 9
# 1 2
# 1 3
# 2 4
# 3 5
# 3 6
# 4 7
# 4 8
# 4 9

 9
 1 2
 1 3
 2 4
 3 5
 3 6
 4 7
 4 8
 4 9


3


## 트리의 독립집합

- 문제 출처: [백준 2213번](https://www.acmicpc.net/problem/2213)

`-` [우수 마을](https://www.acmicpc.net/problem/1949) 문제에 역추적을 더한 문제이다

`-` 여태까지 트리 DP 문제를 잘 해결했다면 이 문제도 쉽게 풀 수 있다

In [5]:
import sys
from collections import defaultdict


def dfs(root, parent, graph, weights):
    for child in graph[root]:
        if child == parent:
            continue
        dfs(child, root, graph, weights)
        if dp[child][1] > dp[child][0]:
            dp[root][0] += dp[child][1]
            track[root][0].append((child, 1))
        else:
            dp[root][0] += dp[child][0]
            track[root][0].append((child, 0))
        dp[root][1] += dp[child][0]
        track[root][1].append((child, 0))


def solution():
    global dp, track
    n = int(input())
    weights = [0] + list(map(int, input().split()))
    dp = [[0 for _ in range(2)] for _ in range(n + 1)]  # dp[x][0]을 x가 루트인 서브 트리에서 x를 제외한 최대 독립 집합이라고 하자 (dp[x][1]은 포함)
    track = [[[] for _ in range(2)] for _ in range(n + 1)]  # track[x][0]은 x가 루트인 서브 트리에서 x는 사용 안할 때 최대 독립 집합에 포함되는 자식 노드
    for u in range(1, n + 1):  # 초깃값
        dp[u][0] = 0
        dp[u][1] = weights[u]
    graph = defaultdict(list)
    for _ in range(n - 1):
        u, v = map(int, input().split())
        graph[u].append(v)
        graph[v].append(u)
    ROOT = 1
    NONE = -1
    dfs(ROOT, NONE, graph, weights)
    answer = []
    if dp[ROOT][1] > dp[ROOT][0]:
        answer.append(ROOT)
        node, is_in = ROOT, 1
    else:
        node, is_in = ROOT, 0
    stack = [(node, is_in)]
    while stack:
        node, is_in = stack.pop()
        for child, is_in in track[node][is_in]:
            if is_in:
                answer.append(child)
            stack.append((child, is_in))
    answer.sort()
    print(max(dp[ROOT]))
    print(*answer)


solution()

# input
# 7
# 10 30 40 10 20 20 70
# 1 2
# 2 3
# 4 3
# 4 5
# 6 2
# 6 7

 7
 10 30 40 10 20 20 70
 1 2
 2 3
 4 3
 4 5
 6 2
 6 7


140
1 3 5 7


## 로스팅하는 엠마도 바리스타입니다

- 문제 출처: [백준 15647번](https://www.acmicpc.net/problem/15647)

`-` 매번 트리를 재구성하고 루트 노드에서 다른 농장들까지 가는 최단 거리의 합을 계산하는 건 $O\left(N^2\right)$이다

`-` 가지고 있는 정보를 재활용하여 루트 노드가 바뀔 때의 변화를 효율적으로 관리하자

`-` $\operatorname{dp}[n]$을 트리에서 $n$번 노드가 루트 노드일 때 루트 노드에서 다른 농장들까지 가는 최단 거리의 합이라고 하자

`-` 그리고 $s[n]$을 $n$번 노드를 루트 노드로 하는 서브 트리에 대해 $n$번 노드에서 자식 노드들까지 가는 최단 거리의 합이라고 하자

`-` 맨 처음엔 루트 노드를 $1$번 노드로 하여 트리를 구성한 뒤 $\operatorname{dp}[n]$을 구해보자

`-` 처음엔 자명하게 $\operatorname{dp}[1] = s[1]$이다

`-` 이제 $1$번 루트 노드의 자식 노드 $c_1, c_2, \dots, c_n$을 고려하자

`-` $\operatorname{dp}[c_1]$을 계산 해보자

`-` 일단 $s[c_1]$을 확보하고 $c_1$의 부모 노드를 거쳐 다른 노드들까지 가는 최단 거리를 더해주면 된다

`-` 그럴러면 $c_1$의 부모 노드에 대해 $c_1$ 쪽의 서브 트리를 제외한 나머지만 고려한 $\operatorname{dp}'[1]$을 알아야 한다

`-` $\operatorname{dp}'[1] = \operatorname{dp}[1] - s[c_1] - d(1, c_1) \cdot \operatorname{size}[c_1]$이다

`-` $\operatorname{size}[n]$은 $n$번 노드를 루트 노드로 하는 서브 트리에 있는 노드 수이다

`-` $d(u, v)$는 $u$번 노드와 $v$번 노드를 잇는 간선의 길이이다

`-` $N - \operatorname{size}[c_1]$이 $c_1$에서 부모 노드를 거쳐서 갈 수 있는 다른 노드들의 수이다 (부모 노드도 포함)

`-` 그럼 $\operatorname{dp}[c_1] = s[c_1] + \operatorname{dp}'[1] + (N - \operatorname{size}[c_1]) \cdot d(1, c_1) = \operatorname{dp}[1] + (N - 2\operatorname{size}[c_1]) \cdot d(1, c_1)$이다

`-` 즉, 부모 노드의 $\operatorname{dp}$값을 알면 그 자식들의 $\operatorname{dp}$값을 $O(1)$에 계산할 수 있다

`-` 노드 개수는 $N$이므로 위 방식의 시간 복잡도는 $O(N)$이다

`-` 참고로 $\operatorname{size}$ 배열과 $s$ 배열은 처음에 DFS를 통해 트리를 구성할 때 계산하면 된다

In [2]:
import sys
from collections import defaultdict

sys.setrecursionlimit(10**6)


def dfs(tree, node, parent, dist, size, subtree_dp):
    size[node] = 1
    for child in tree[node]:
        if child == parent:
            continue
        dfs(tree, child, node, dist, size, subtree_dp)
        size[node] += size[child]
        subtree_dp[node] += subtree_dp[child] + dist[node, child] * size[child]


def tree_dp(tree, node, parent, dist, size, dp):
    dp[node] = dp[parent] + (N - 2 * size[node]) * dist[node, parent]
    for child in tree[node]:
        if child == parent:
            continue
        tree_dp(tree, child, node, dist, size, dp)


def solution():
    global N
    N = int(input())
    tree = defaultdict(list)
    dist = {}
    root = 1
    none = 0
    size = [0] * (N + 1)
    subtree_dp = [0] * (N + 1)
    dp = [0] * (N + 1)
    for _ in range(N - 1):
        u, v, d = map(int, input().split())
        tree[u].append(v)
        tree[v].append(u)
        dist[u, v] = dist[v, u] = d
    dist[none, root] = dist[root, none] = 0
    dfs(tree, root, none, dist, size, subtree_dp)
    dp[none] = subtree_dp[root]
    tree_dp(tree, root, none, dist, size, dp)
    for u in range(1, N + 1):
        print(dp[u])


solution()

# input
# 10
# 1 2 1
# 2 3 1
# 2 4 1
# 4 7 1
# 4 8 1
# 4 5 1
# 1 6 1
# 6 9 1
# 6 10 1

 10
 1 2 1
 2 3 1
 2 4 1
 4 7 1
 4 8 1
 4 5 1
 1 6 1
 6 9 1
 6 10 1


19
17
25
19
27
23
27
27
31
31


`-` $\operatorname{dp}[c_1] = \operatorname{dp}[1] + (N - 2\operatorname{size}[c_1]) \cdot d(1, c_1)$의 의미를 생각해보면 다음과 같았다

`-` 루트 노드를 기준으로 생각했을 때 $c_1$이 루트 노드가 되면 $1$번이 루트 노드였을 때 존재했던 $1 \to c_1$ 경로가 사라지므로 루트 노드에서 다른 노드로 가는 최단 거리의 합이 $\operatorname{size}[c_1] \cdot d(1, c_1)$만큼 감소된다

`-` 그런데 $c_1$이 루트 노드가 되면 $1$번 노드가 루트 노드였을 때 없었던 $c_1 \to 1$ 경로가 새롭게 추가되므로 최단 거리의 합이 $(N - \operatorname{size}[c_1]) \cdot d(c_1, 1)$만큼 증가한다

`-` 따라서 위의 점화식이 도출된다

`-` 이렇게 효율적으로 루트 노드를 변경하여 원하는 것을 계산하는 방식을 rerooting이라고 하더라